In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
from src.backbone import Backbone
from src.rpn import RPN_Head, RegionProposalNetwork

backbone = Backbone().to(device)
rpn_head = RPN_Head(in_channels=1024, mid_channels=512).to(device)

In [3]:
from pathlib import Path
import torch

# checkpoint_dir = Path("/content/drive/MyDrive/faster_rcnn/checkpoints")
checkpoint_dir = Path("checkpoints")

existing_checkpoints = sorted(checkpoint_dir.glob("step1_epoch_*.pt"),
                              key = lambda p : int(p.stem.split("_epoch_")[1]))

print(f"Existing checkpoints: {existing_checkpoints}")

if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")

    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    backbone.load_state_dict(checkpoint['backbone_state_dict'])
    rpn_head.load_state_dict(checkpoint['rpn_head_state_dict'])



Existing checkpoints: [PosixPath('checkpoints/step1_epoch_1.pt'), PosixPath('checkpoints/step1_epoch_2.pt'), PosixPath('checkpoints/step1_epoch_3.pt'), PosixPath('checkpoints/step1_epoch_4.pt'), PosixPath('checkpoints/step1_epoch_5.pt'), PosixPath('checkpoints/step1_epoch_6.pt'), PosixPath('checkpoints/step1_epoch_7.pt'), PosixPath('checkpoints/step1_epoch_8.pt'), PosixPath('checkpoints/step1_epoch_9.pt'), PosixPath('checkpoints/step1_epoch_10.pt')]
Loading checkpoint: checkpoints/step1_epoch_10.pt


In [4]:
from src.backbone import backbone_transform
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

transform = backbone_transform

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

combined_train_img_paths = voc2007_img_paths_train + voc2012_img_paths_train

train_dataloader = create_voc_dataloader(img_paths_list=combined_train_img_paths, transform=transform, batch_size=2, shuffle=True)

In [5]:
batch_imgs, batch_boxes, batch_labels, img_sizes_before_pad = next(iter(train_dataloader))

rpn_network = RegionProposalNetwork(rpn_head=rpn_head).to(device)

with torch.inference_mode():
    backbone.eval()
    rpn_head.eval()
    rpn_network.eval()

    batch_feature_maps = backbone(batch_imgs.to(device))
    print(f"batch_feature_maps.shape: {batch_feature_maps.shape}")
    batch_scores, batch_proposals = rpn_network(batch_feature_maps, batch_imgs.shape[2], batch_imgs.shape[3], img_sizes_before_pad)
    

batch_feature_maps.shape: torch.Size([2, 1024, 57, 38])


In [6]:
import torch
import torch.nn as nn

class RoIPool(nn.Module):
    def __init__(self, output_size, pooling_mode="loop"):
        super().__init__()
        self.output_size = output_size
        self.pooling_mode = pooling_mode
        self.adaptive_pool = nn.AdaptiveMaxPool2d(output_size)

    def forward(self, feature_maps, batch_proposals, batch_img_height, batch_img_width):
        channels = feature_maps.shape[1]
        feat_height, feat_width = feature_maps.shape[2], feature_maps.shape[3]
        out_h, out_w = self.output_size

        stride_x = batch_img_width // feat_width
        stride_y = batch_img_height // feat_height

        pooled_batch = []
        for feature_map, proposals in zip(feature_maps, batch_proposals):
            if proposals.shape[0] == 0:
                # No proposals survived for this image — keep a well-shaped empty tensor
                pooled_batch.append(feature_map.new_empty((0, channels, out_h, out_w)))
                continue

            projected = self._project_to_feature_map(proposals, feat_height, feat_width, stride_x, stride_y)

            pooled_proposals = []
            for proposal_coords in projected:
                roi_feature_map = self._select_roi_feature_map(feature_map, proposal_coords)
                
                if self.pooling_mode == "adaptive":
                    pooled_proposals.append(self._max_pool_roi_adaptive(roi_feature_map))
                else:
                    pooled_proposals.append(self._max_pool_roi(roi_feature_map))

            pooled_batch.append(torch.stack(pooled_proposals, dim=0))  # [N_i, C, out_h, out_w]

        return pooled_batch

    def _project_to_feature_map(self, proposals, feat_height, feat_width, stride_x, stride_y):
        x1 = proposals[:, 0]
        y1 = proposals[:, 1]
        x2 = proposals[:, 2]
        y2 = proposals[:, 3]

        fx1 = torch.round(x1 / stride_x)
        fy1 = torch.round(y1 / stride_y)
        fx2 = torch.round(x2 / stride_x)
        fy2 = torch.round(y2 / stride_y)

        # Convert to long type for holding larger values and prevent overflow due to decimal double
        fx1 = fx1.long()
        fy1 = fy1.long()
        fx2 = fx2.long()
        fy2 = fy2.long()

        fx1 = torch.clamp(fx1, 0, feat_width - 1)
        fx2 = torch.clamp(fx2, 0, feat_width - 1)
        fy1 = torch.clamp(fy1, 0, feat_height - 1)
        fy2 = torch.clamp(fy2, 0, feat_height - 1)

        return torch.stack([fx1, fy1, fx2, fy2], dim=-1)  # [N, 4]

    def _select_roi_feature_map(self, feature_map, proposal_coords):
        fx1, fy1, fx2, fy2 = proposal_coords

        return feature_map[:, fy1:fy2 + 1, fx1:fx2 + 1]  # [C, roi_h, roi_w]

    def _max_pool_roi(self, roi_feature_map):
        channels, roi_h, roi_w = roi_feature_map.shape
        out_h, out_w = self.output_size  # output_size is (H, W) of the pooled map
        device = roi_feature_map.device

        # Bin boundaries per axis (each axis uses its own bin count): floor for
        # start, ceil for end so every bin covers >= 1 pixel even when the ROI
        # is smaller than the number of bins along that axis.
        h_idx = torch.arange(out_h, device=device)
        h_starts = torch.clamp(torch.floor(h_idx * roi_h / out_h).long(), 0, roi_h)
        h_ends = torch.clamp(torch.ceil((h_idx + 1) * roi_h / out_h).long(), 0, roi_h)

        w_idx = torch.arange(out_w, device=device)
        w_starts = torch.clamp(torch.floor(w_idx * roi_w / out_w).long(), 0, roi_w)
        w_ends = torch.clamp(torch.ceil((w_idx + 1) * roi_w / out_w).long(), 0, roi_w)

        pooled = roi_feature_map.new_empty((channels, out_h, out_w))
        for ph in range(out_h):
            for pw in range(out_w):
                bin_region = roi_feature_map[:, h_starts[ph]:h_ends[ph], w_starts[pw]:w_ends[pw]]
                pooled[:, ph, pw] = bin_region.amax(dim=(1, 2))  # max over the bin, per channel

        return pooled  # [C, out_h, out_w]

    def _max_pool_roi_adaptive(self, roi_feature_map):
        # Use adaptive pooling to directly get the desired output size, adding 1 dim for batch_processing
        pooled = self.adaptive_pool(roi_feature_map.unsqueeze(0))
        return pooled.squeeze(0)  # [C, out_h, out_w]


In [7]:
roi_pool = RoIPool(output_size=(7, 7), pooling_mode="adaptive").to(device)

with torch.inference_mode():
    roi_pool.eval()
    pooled_batch = roi_pool(batch_feature_maps, batch_proposals, batch_imgs.shape[2], batch_imgs.shape[3])

print(f"batch size: {len(pooled_batch)} (expected {len(batch_proposals)})")
for i, (pooled, proposals) in enumerate(zip(pooled_batch, batch_proposals)):
    print(f"  img {i}: pooled {tuple(pooled.shape)}  |  n_proposals={proposals.shape[0]}  |  device={pooled.device}")

    # Sanity: max-pooled values must stay within each source feature map's range
    fmap = batch_feature_maps[i]
    assert pooled.shape == (proposals.shape[0], batch_feature_maps.shape[1], 7, 7)
    if pooled.numel() > 0:
        assert pooled.min() >= fmap.min() and pooled.max() <= fmap.max(), "pooled value out of source range"

print("OK")

batch size: 2 (expected 2)
  img 0: pooled (225, 1024, 7, 7)  |  n_proposals=225  |  device=cuda:0
  img 1: pooled (294, 1024, 7, 7)  |  n_proposals=294  |  device=cuda:0
OK
